# Validation Suite — Steady-State Sampler

Three independent checks that `sample_steady_state` (the direct Poisson–Beta sampler of the
telegraph steady state) reproduces the known analytical results of the two-state model.

1. **Convergence to ODE** — as $n_{rep} \to \infty$ the sample moments approach the steady state of
   the moment ODEs (`solve_ode_moments` at large $t$).
2. **Analytical moments** — at large $n_{rep}$ the five moments match their closed-form expressions.
3. **Edge cases** — constitutive gene ($k_{off}=0 \Rightarrow$ Poisson) and the degenerate
   $k_{deg}=0$ (no steady state).

Each case prints the computed quantities next to their theoretical reference and shows a supporting
figure.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import plotly.graph_objects as go
from scipy import stats

from src.steady_state import sample_steady_state
from src.ode_moments import solve_ode_moments

np.random.seed(0)   

In [2]:
G_COLOR = "#3B82F6"     # gene state
R_COLOR = "#EF4444"     # RNA count
COV_COLOR = "#8B5CF6"   # covariance


def _rgba(hex_color, alpha):
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r}, {g}, {b}, {alpha})"


def analytic_moments(k_on, k_off, k_syn, k_deg) -> dict:
    mu_G = k_on / (k_on + k_off)
    mu_R = (k_syn / k_deg) * mu_G
    cov_RG = k_syn * mu_G * (1.0 - mu_G) / (k_on + k_off + k_deg)
    var_R = mu_R + (k_syn / k_deg) * cov_RG
    return {
        "mu_G": mu_G,
        "mu_R": mu_R,
        "sigma_G": np.sqrt(mu_G * (1.0 - mu_G)),
        "sigma_R": np.sqrt(var_R),
        "cov_RG": cov_RG,
    }

---
## Case 1 — Convergence to ODE Predictions ($n_{rep} \to \infty$)

**Setup:** `k_on=0.5, k_off=1.0, k_syn=20.0, k_deg=1.0`.

In [3]:
kin = dict(k_on=0.5, k_off=1.0, k_syn=20.0, k_deg=1.0)

_, y = solve_ode_moments(**kin, t0=0.0, g0=0, r0=0, t_end=200.0)
mu_R_ode = y[1, -1]

n_reps = [10, 100, 1000, 10000, 100000]
errors = []
for n_rep in n_reps:
    m = sample_steady_state(**kin, n_rep=n_rep)
    err = abs(m["mu_R"] - mu_R_ode)
    errors.append(err)
    print(f"n_rep = {n_rep:6d}  |  |sample mu_R - ODE mu_R| = {err:.4f}")

print(f"\nODE steady-state mu_R = {mu_R_ode:.4f}")


n_rep =     10  |  |sample mu_R - ODE mu_R| = 3.8667
n_rep =    100  |  |sample mu_R - ODE mu_R| = 0.5833
n_rep =   1000  |  |sample mu_R - ODE mu_R| = 0.1917
n_rep =  10000  |  |sample mu_R - ODE mu_R| = 0.1009
n_rep = 100000  |  |sample mu_R - ODE mu_R| = 0.0145

ODE steady-state mu_R = 6.6667


In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=n_reps, y=errors, mode="lines+markers",
    line=dict(color=R_COLOR, width=2), name="|sample - ODE| ⟨R⟩",
))
guide = errors[0] * np.sqrt(n_reps[0] / np.array(n_reps, float))   # 1/sqrt(n) reference
fig.add_trace(go.Scatter(
    x=n_reps, y=guide, mode="lines",
    line=dict(color="gray", width=2, dash="dash"), name="∝ 1/√n_rep",
))
fig.update_layout(
    template="plotly_white", width=700, height=450,
    title="<b>Case 1 — Sampler → ODE Convergence vs n_rep</b>",
    margin=dict(l=85, r=30, t=70, b=50), hovermode="x",
    xaxis_title="n_rep", yaxis_title="abs error in ⟨R⟩",
)
fig.update_xaxes(type="log")
fig.update_yaxes(type="log")
fig.show()

---
## Case 2 — Match Analytical Moments

**Setup:** `k_on=0.5, k_off=1.0, k_syn=20.0, k_deg=1.0`, large `n_rep = 200000`.

**Scientific logic:** the steady-state moments have closed forms

$$\mu_G=\frac{k_{on}}{k_{on}+k_{off}},\qquad
\mu_R=\frac{k_{syn}}{k_{deg}}\,\mu_G,\qquad
C_{RG}=\frac{k_{syn}\,\mu_G(1-\mu_G)}{k_{on}+k_{off}+k_{deg}},$$

$$\sigma_G=\sqrt{\mu_G(1-\mu_G)},\qquad
\sigma_R=\sqrt{\mu_R+\tfrac{k_{syn}}{k_{deg}}\,C_{RG}}.$$

**Output:** each sampler moment next to its closed-form value, with the absolute difference.

In [5]:
kin = dict(k_on=0.5, k_off=1.0, k_syn=20.0, k_deg=1.0)
n_rep = 200_000

theory = analytic_moments(**kin)
sample = sample_steady_state(**kin, n_rep=n_rep)

keys = ["mu_G", "mu_R", "sigma_G", "sigma_R", "cov_RG"]
print(f"{'moment':8s} {'analytic':>10s} {'sample':>10s} {'abs diff':>10s}")
for key in keys:
    print(f"{key:8s} {theory[key]:10.4f} {sample[key]:10.4f} {abs(sample[key] - theory[key]):10.4f}")

moment     analytic     sample   abs diff
mu_G         0.3333     0.3317     0.0016
mu_R         6.6667     6.6277     0.0389
sigma_G      0.4714     0.4708     0.0006
sigma_R      6.4979     6.4767     0.0212
cov_RG       1.7778     1.7666     0.0111


In [6]:
var_color = {"mu_G": G_COLOR, "sigma_G": G_COLOR,
             "mu_R": R_COLOR, "sigma_R": R_COLOR, "cov_RG": COV_COLOR}
solid = [var_color[k] for k in keys]                  # analytic = full colour
light = [_rgba(var_color[k], 0.45) for k in keys]     # sample = lighter

fig = go.Figure()
fig.add_trace(go.Bar(
    x=keys, y=[theory[k] for k in keys], name="analytic",
    marker_color=solid, marker_line=dict(width=1, color="rgba(0, 0, 0, 0.3)"),
))
fig.add_trace(go.Bar(
    x=keys, y=[sample[k] for k in keys], name="sample",
    marker_color=light, marker_line=dict(width=1, color="rgba(0, 0, 0, 0.3)"),
))
fig.update_layout(
    template="plotly_white", width=750, height=450, barmode="group",
    title="<b>Case 2 — Sampler Moments vs Analytical</b>",
    margin=dict(l=85, r=30, t=70, b=50), hovermode="x",
    xaxis_title="moment", yaxis_title="value",
)
fig.show()

---
## Case 3 — Edge Cases

**$k_{off}=0$ (constitutive gene):** the gene is permanently ON, so the steady state is a pure
birth–death process: $G\equiv 1$ and $R \sim \text{Poisson}(k_{syn}/k_{deg})$
(mean = variance, Fano = 1).

**$k_{deg}=0$ (no degradation):** RNA accumulates without bound, so no stationary distribution
exists — `sample_steady_state` must reject this input.

In [7]:
# --- Edge case A: k_off = 0  ->  R ~ Poisson(k_syn / k_deg) ---
k_syn, k_deg = 10.0, 1.0
mu_theory = k_syn / k_deg
edge = sample_steady_state(k_on=0.5, k_off=0.0, k_syn=k_syn, k_deg=k_deg, n_rep=100_000)
fano = edge["sigma_R"] ** 2 / edge["mu_R"]

print("Edge A (k_off=0, constitutive ON):")
print(f"  mu_G = {edge['mu_G']:.4f}   (Poisson expects 1)")
print(f"  mu_R = {edge['mu_R']:.3f}   (Poisson expects {mu_theory:.0f})")
print(f"  Fano = {fano:.3f}   (Poisson expects 1)")

# --- Edge case B: k_deg = 0  ->  no steady state ---
try:
    sample_steady_state(k_on=0.5, k_off=0.5, k_syn=10.0, k_deg=0.0, n_rep=10)
except ValueError as err:
    print(f"\nEdge B (k_deg=0): sample_steady_state raised ValueError -> {err}")

Edge A (k_off=0, constitutive ON):
  mu_G = 1.0000   (Poisson expects 1)
  mu_R = 9.994   (Poisson expects 10)
  Fano = 0.993   (Poisson expects 1)

Edge B (k_deg=0): sample_steady_state raised ValueError -> Steady state is undefined for k_deg = 0 (RNA grows unbounded).


In [8]:
R = edge["R_samples"]
k = np.arange(R.min(), R.max() + 1)
pmf = stats.poisson.pmf(k, mu_theory)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=R, histnorm="probability",
    xbins=dict(start=R.min() - 0.5, end=R.max() + 0.5, size=1),
    marker_color=_rgba(R_COLOR, 0.55), name="sampler (k_off=0)",
))
fig.add_trace(go.Scatter(
    x=k, y=pmf, mode="lines+markers",
    line=dict(color=R_COLOR, width=2, dash="dash"), name=f"Poisson(μ={mu_theory:.0f})",
))
fig.update_layout(
    template="plotly_white", width=800, height=450, bargap=0.05, hovermode="x",
    title="<b>Case 3 — Constitutive Gene: Sampler vs Poisson PMF</b>",
    margin=dict(l=85, r=30, t=70, b=50),
    xaxis_title="RNA count R", yaxis_title="probability",
)
fig.show()